In [ ]:
#Reference
#https://github.com/fnusatvik07/agent-builder-thinkinginglanggraph/blob/main/workbooks/06_llm_augmentations.ipynb

In [ ]:
!pip install langchain langchain-groq langgraph python-dotenv ipython --quiet

In [11]:
import os
from dotenv import load_dotenv
load_dotenv('/content/.env.txt')
print("Keys loaded:", [k for k in ("GROQ_API_KEY","OPENAI_API_KEY", "TAVILY_API_KEY", "SERPAPI_API_KEY") if os.getenv(k)])


Keys loaded: ['GROQ_API_KEY', 'TAVILY_API_KEY', 'SERPAPI_API_KEY']


In [12]:
# Initialize Groq model
from langchain.chat_models import init_chat_model

In [13]:
llm = init_chat_model(
    "groq:llama-3.3-70b-versatile",
    temperature=0
)

print("LLM ready")

LLM ready


**1 with_structured_output:**

>Pydantic returns an object

>TypedDict returns a dictionary:


**When not to reach for structured output**

Long form generation (drafting an email body, writing a report section)

When your schema would have a giant str field that is the whole answer anyway

In [18]:
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    search_query: str = Field(description="Query optimized for web search.")
    justification: str = Field(description="Why this query is relevant.")

structured_llm = llm.with_structured_output(SearchQuery)
out = structured_llm.invoke("How does insults, silent struggles,leads to unhapiness?")
print("type:        ", type(out).__name__)
print("search_query:", out.search_query)
print("justification:", out.justification)

type:         SearchQuery
search_query: insults silent struggles unhappiness
justification: To understand how insults and silent struggles can lead to unhappiness, a search query is necessary to find relevant information on the psychological effects of insults and the impact of silent struggles on mental health.


In [19]:
from typing import TypedDict, Literal

class Triage(TypedDict):
    intent: Literal["question", "bug", "billing", "feature", "complex"]
    urgency: Literal["low", "medium", "high", "critical"]
    one_line_summary: str

triage_llm = llm.with_structured_output(Triage)
triage = triage_llm.invoke("My subscription was charged twice this month and I need a refund urgently.")
print(triage)
print("is a regular dict at runtime:", type(triage).__name__)


{'intent': 'billing', 'one_line_summary': 'Double charge on subscription', 'urgency': 'high'}
is a regular dict at runtime: dict


**Literal with Structured Output**

Literal   → restrict allowed values (great for routing)

In [20]:
from typing import Literal
from pydantic import BaseModel

class Route(BaseModel):
    next_step: Literal["search", "math", "end"]

In [26]:
router = llm.with_structured_output(Route)

query = "What is 2+2?"

result = router.invoke(
    f"Route this query. Use 'math' for calculations and 'search' for knowledge questions.\n{query}"
)

if result.next_step == "math":
    print(5*7)

else:
    print("Search Node")

35


**2.Tool calling**

In [27]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

llm_with_tools = llm.bind_tools([multiply])
msg = llm_with_tools.invoke("What is 2 times 3?")
print("content:    ", msg.content)
print("tool_calls: ", msg.tool_calls)

'''the docstrings and type hints become the tool schemas for LLM
tool_calls is a list of dicts: each has name, args, and an id that you echo back in the tool response.
'''

content:     
tool_calls:  [{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'xwzykhrq1', 'type': 'tool_call'}]


In [29]:
for tc in msg.tool_calls:
    result = multiply(**tc["args"]) #unpacks the dictionary into keyword arguments
    # then build a ToolMessage(content=str(result), tool_call_id=tc["id"])
    # and send it back to the model for the final answer.
    print(result)


6


**3.Short-term memory**

Each llm.invoke(...) is a fresh request.

In [30]:
from langchain.messages import HumanMessage, AIMessage

# A two-turn conversation by hand to show how the message list grows.
history = [
    HumanMessage(content="Hi, my name is Jolly."),
    AIMessage(content="Nice to meet you, Jolly!"),
    HumanMessage(content="What did I say my name was?"),
]
reply = llm.invoke(history)
print(reply.content)


You said your name is Jolly.


In [34]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq


# State
class State(TypedDict):
    messages: Annotated[list, add_messages]


# Node
def chatbot(state: State):
    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }


# Graph
graph = StateGraph(State)

graph.add_node("chatbot", chatbot)

graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()


# Run 1
result = app.invoke({
    "messages": [
        HumanMessage(content="Hi, my name is Jolly")
    ]
})

print(result["messages"][-1].content)

# Run 2 (history included)
result = app.invoke({
    "messages": result["messages"] + [
        HumanMessage(content="What is my name?")
    ]
})

print(result["messages"][-1].content)

Hello Jolly, it's nice to meet you. Is there something I can help you with or would you like to chat?
Your name is Jolly.


In [36]:
print(result["messages"])

[HumanMessage(content='Hi, my name is Jolly', additional_kwargs={}, response_metadata={}, id='24af78cc-5f6d-4439-8b8a-493bed66592e'), AIMessage(content="Hello Jolly, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 42, 'total_tokens': 69, 'completion_time': 0.052979356, 'completion_tokens_details': None, 'prompt_time': 0.003931032, 'prompt_tokens_details': None, 'queue_time': 0.164389318, 'total_time': 0.056910388}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e87c9-14a4-7052-bde4-2d7a7d451536-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 27, 'total_tokens': 69}), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='9